# Notebook 10 — Matplotlib and Seaborn for Scientific Plotting

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 10 of 20  
**Prerequisites:** Notebook 09  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Visualization is the primary mode of biological discovery. A PCA plot revealing
batch separation, a volcano plot identifying differentially expressed genes, a heatmap
showing co-expression clusters — these are the figures that appear in papers and
drive experimental decisions.

This notebook builds the three core plots in `compbio_utils/plotting.py` that every
later module will reuse.

---
## Step 2 — Intuition

Matplotlib is the rendering engine: it gives you full control at the cost of verbosity.
Seaborn is a statistical plotting layer on top of Matplotlib: it produces aesthetically
good defaults for grouped, statistical data with less code.

Use Seaborn when the data fits its grammar (grouped comparisons, distributions);
drop to raw Matplotlib when you need precise layout control or custom annotations.

---
## Step 3 — Biological Background

**The three most-used plots in transcriptomics:**

| Plot | X axis | Y axis | Colour/Size | Reveals |
|------|--------|--------|-------------|----------|
| PCA scatter | PC1 | PC2 | Sample condition/batch | Batch effects, major sources of variance |
| Volcano plot | log2(fold-change) | -log10(p-value) | Significance threshold | Differentially expressed genes |
| Expression heatmap | Samples | Genes | Expression intensity | Co-expression clusters |

All three appear in the final portfolio artifact for Module 01.

---
## Step 4 — Mathematical Explanation

**Volcano plot axes:**

- X: $\log_2 \text{FC} = \log_2\left(\frac{\bar{x}_{\text{tumour}}}{\bar{x}_{\text{normal}}}\right)$
- Y: $-\log_{10}(p)$ where $p$ is the two-sample t-test p-value

Points in the upper-left and upper-right quadrants (high $|-\log_{10}p|$ and high
$|\log_2 \text{FC}|$) are differentially expressed genes of interest.

---
## Step 5 — Computational Explanation

**Matplotlib object model:**

```
Figure                   ← the whole canvas
  └── Axes               ← one plot panel
        ├── Axis          ← x-axis, y-axis
        ├── Artists       ← lines, patches, text, images
        └── Legend
```

Always use the explicit `fig, ax = plt.subplots()` form — never the implicit `plt.plot()`
form in library code. The explicit form is debuggable and composable.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats

print(f"matplotlib {plt.matplotlib.__version__}")
print(f"seaborn    {sns.__version__}")

# Set style once at the top of a notebook
sns.set_theme(style="whitegrid", palette="colorblind", font_scale=1.1)

rng = np.random.default_rng(42)

In [ ]:
# Cell 6.1 — Generate synthetic expression data (2 conditions)
n_genes = 300
n_tumour, n_normal = 6, 6

tumour = rng.exponential(scale=5.0, size=(n_genes, n_tumour))
normal = rng.exponential(scale=5.0, size=(n_genes, n_normal))

# Make 30 genes differentially expressed
tumour[:15] *= 4    # up-regulated in tumour
tumour[15:30] /= 4  # down-regulated in tumour

X = np.hstack([tumour, normal])
labels = ["tumour"]*n_tumour + ["normal"]*n_normal
gene_ids = [f"GENE_{i:04d}" for i in range(n_genes)]

In [ ]:
# Cell 6.2 — PCA scatter plot
def pca_scatter(
    X: np.ndarray,
    labels: list[str],
    title: str = "PCA",
    ax=None,
) -> plt.Axes:
    """
    PCA scatter plot coloured by sample label.
    Uses compbio_utils.stats.pca_svd if available, else sklearn.
    """
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    Z = pca.fit_transform(X.T)  # samples as rows
    evr = pca.explained_variance_ratio_

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))

    palette = sns.color_palette("colorblind", n_colors=len(set(labels)))
    for i, label in enumerate(sorted(set(labels))):
        mask = [l == label for l in labels]
        ax.scatter(Z[mask, 0], Z[mask, 1],
                   label=label, color=palette[i], s=60, alpha=0.85, edgecolors="white")

    ax.set_xlabel(f"PC1 ({evr[0]:.1%} variance)")
    ax.set_ylabel(f"PC2 ({evr[1]:.1%} variance)")
    ax.set_title(title)
    ax.legend(frameon=False)
    return ax

fig, ax = plt.subplots(figsize=(6, 5))
pca_scatter(np.log2(X + 1), labels, title="PCA — log2(counts+1)", ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6.3 — Volcano plot
def volcano_plot(
    log2fc: np.ndarray,
    pvalues: np.ndarray,
    gene_ids: list[str],
    fc_threshold: float = 1.0,
    pval_threshold: float = 0.05,
    n_labels: int = 5,
    ax=None,
) -> plt.Axes:
    """Volcano plot: log2FC vs -log10(p-value)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))

    neg_log_p = -np.log10(np.clip(pvalues, 1e-300, 1))
    sig = (np.abs(log2fc) > fc_threshold) & (pvalues < pval_threshold)

    ax.scatter(log2fc[~sig], neg_log_p[~sig], s=8, alpha=0.4, color="lightgray")
    ax.scatter(log2fc[sig],  neg_log_p[sig],  s=12, alpha=0.8,
               c=np.where(log2fc[sig] > 0, "tomato", "steelblue"))

    ax.axvline(-fc_threshold, color="gray", lw=0.8, ls="--")
    ax.axvline( fc_threshold, color="gray", lw=0.8, ls="--")
    ax.axhline(-np.log10(pval_threshold), color="gray", lw=0.8, ls="--")

    # Label top genes
    top_idx = np.argsort(neg_log_p)[::-1][:n_labels]
    for idx in top_idx:
        ax.annotate(gene_ids[idx], (log2fc[idx], neg_log_p[idx]),
                    fontsize=7, ha="center", va="bottom")

    ax.set_xlabel("log₂(Fold Change)")
    ax.set_ylabel("-log₁₀(p-value)")
    ax.set_title(f"Volcano plot  ({sig.sum()} DE genes)")
    return ax

# Compute fold change and p-values
log2fc = np.log2(tumour.mean(axis=1) + 1) - np.log2(normal.mean(axis=1) + 1)
pvals = np.array([
    sp_stats.ttest_ind(tumour[i], normal[i]).pvalue
    for i in range(n_genes)
])

fig, ax = plt.subplots(figsize=(7, 5))
volcano_plot(log2fc, pvals, gene_ids, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6.4 — Expression heatmap (top 30 DE genes)
def expression_heatmap(
    df: pd.DataFrame,
    title: str = "Expression Heatmap",
    figsize: tuple = (10, 6),
) -> plt.Figure:
    """Seaborn clustermap of a genes×samples expression DataFrame."""
    g = sns.clustermap(
        df,
        cmap="RdBu_r",
        standard_scale=0,   # z-score rows
        figsize=figsize,
        yticklabels=True,
        xticklabels=True,
    )
    g.fig.suptitle(title, y=1.02)
    return g.fig

# Select top 30 DE genes by p-value
top30_idx = np.argsort(pvals)[:30]
df_top = pd.DataFrame(
    np.log2(X[top30_idx] + 1),
    index=[gene_ids[i] for i in top30_idx],
    columns=labels,
)
expression_heatmap(df_top, title="Top 30 DE genes — log2(counts+1)")
plt.show()

---
## Step 7 — Visualization Notes

The visualizations above *are* Step 7 — they are the deliverables of this notebook.
Ensure all three plots render correctly and are publication-ready (correct axis labels,
title, legend, no overlapping text).

---
## Step 8 — Exercises

1. Add `pca_scatter`, `volcano_plot`, and `expression_heatmap` to `compbio_utils/plotting.py`.
   Each must accept an optional `ax` parameter so they can be embedded in multi-panel figures.
2. Create a 3-panel publication figure (PCA | Volcano | Heatmap) saved to
   `portfolio/module01_expression_overview.png` at 300 DPI.
3. Add multiple-testing correction to `volcano_plot` using
   `statsmodels.stats.multitest.multipletests` (Benjamini-Hochberg). How does the
   number of significant genes change?
4. Why does the heatmap use `standard_scale=0` (z-score rows) instead of raw counts?

---
## Quiz — Active Recall

1. What is the difference between `fig, ax = plt.subplots()` and `plt.plot()`? Why prefer the first?
2. Why are both axes of a volcano plot on log scale?
3. What does `standard_scale=0` do in a seaborn clustermap?
4. You have 10,000 t-tests. The raw p-value threshold is 0.05. How many false positives
   do you expect without correction?
5. What is a 'colorblind-safe' palette and why does it matter for publication figures?

---
## Reflection

**Date completed:** ____________________

> *[Are all three plots in compbio_utils/plotting.py? Is the portfolio figure saved? Which plot was hardest to implement correctly?]*

---
*Next: `11_scipy_optimization.ipynb`*